Julia Kang - jkang613@gmail.com - AMLI 2019 - Mills College

In [ ]:
import pandas as pd
import numpy as np
import multiprocessing as mp
import random
import datetime
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import linear_model
# from sklearn import model_selection
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier
from math import sqrt #for RMSE


## Model: SGDClassifier Model

**Logistic vs SGDClassifier:** Logistic Regression by default uses Gradient Descent and as such it would be better to use SGD Classifier on larger data sets. One another reason you might want to use SGD Classifier is, logistic regression, in its vanilla sklearn form, won?t work if you can?t hold the dataset in RAM but SGD will still work.

For sufficiently large datasets, it is best to implement SGD Classifier instead of Logistic Classifier to produce similar results in much less time.

In [ ]:
# raw_df = pd.read_csv('numeric_rand_sample.csv')
raw_df = pd.read_csv('choice_features.csv')
raw_df = raw_df.drop('Unnamed: 0', axis = 1) 

raw_df.head()
# len(sub_df)

In [ ]:
FEATURES = []

for col in raw_df.columns:
    if np.issubdtype(raw_df[col].dtypes , np.number):
        FEATURES.append(col)

FEATURES.remove('click')
TARGET = 'click'

In [ ]:
# split the data into test and train
train, test  = train_test_split(
  raw_df,                        # split our digits dataframe
  stratify=raw_df[TARGET],       # stratify by the 'click' column   
  test_size=0.2,                # 20% of the data should be held for testing
  random_state=42               # hard-coded random state for repeatability in the example
)

train.head()

In [ ]:
train_features = np.array(train[FEATURES])
train_labels = np.array(train[TARGET])
test_features = np.array(test[FEATURES])
test_labels = np.array(test[TARGET])

In [ ]:
sgd_classifier = linear_model.SGDClassifier(
    loss='log',
    random_state=2, # Specifying random state allow us to get repeatable outcomes
    tol=1e-3, 
    max_iter=500,
    alpha=0.001
) 
 
sgd_classifier.fit(train_features, train_labels.ravel())

# sgd_classifier.partial_fit(train_features, train_labels.ravel(), classes = [0,1]) #partial_fit SUCKED

In [ ]:
# get the predictions using our model
sgd_predictions = sgd_classifier.predict(test_features)

# we can use the scores to set thresholds on the predictions made
sgd_scores = sgd_classifier.decision_function(test_features)
RMSE = sqrt(metrics.mean_squared_error(test_labels, sgd_predictions))
print(("Model using SGDClassifier has an RMSE of {} ").format(RMSE))
# 0.4158645213482616 - base
# 0.41513549509867104 - loss = log

LL = sqrt(metrics.log_loss(test_labels, sgd_predictions))
print(("Model using SGDClassifier has an LL of {} ").format(LL))
# Model using SGDClassifier has an LL of 2.425344088859301 - Using Ryan's Data

In [ ]:
# # loop through different C (alpha) values to see which gives us the largest AUC value
# alphas = [1e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1]

# LLS = []
# AUCS = []

# for a in alphas:
#     sgd_classifier = linear_model.SGDClassifier(
#         alpha = a,
#         loss='log',
#         random_state=2, # Specifying random state allow us to get repeatable outcomes
#         tol=1e-3, 
#         max_iter=500
#     ) 

#     sgd_classifier.fit(train_features, train_labels.ravel())
#     sgd_predictions = sgd_classifier.predict(test_features)
#     sgd_ll = sqrt(metrics.log_loss(test_labels, sgd_predictions))
#     LLS.append(sgd_ll)
#     sgd_fpr, sgd_tpr, sgd_thresholds = metrics.roc_curve(test_labels, sgd_predictions)
#     sgd_roc_auc = metrics.auc(sgd_fpr, sgd_tpr)
#     AUCS.append(sgd_roc_auc)
    
#     print(("Model where alpha is {} is completed").format(a))

# print(LLS)

# # plot to visiual the different AUC in relation to alpha 
# plt.plot(AUCS)
# plt.xticks(range(len(alphas)), alphas)
# plt.ylabel('AUC')
# plt.xlabel('alpha')
# plt.show()